In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
!pip install optuna
!pip install flowio

!pip install git+https://github.com/enricodidoli-source/memory-save.git
!pip install git+https://github.com/enricodidoli-source/seed-manager.git

import os

os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'  # For CuDNN ops

import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import pickle
import time
import yaml
import tensorflow as tf
from tensorflow import keras
import glob


from memory_utils import clear_RAM, MemorySave
from seed_manager import SeedManager

In [ ]:

#fixed_path = '/content/drive/MyDrive/BinCellCNN/'
fixed_path = 'C:/Users/admin/BinCellCNN/'

whole_exp = 'clinical_try_1'
#whole_exp = 'clinical_try_1'

if fixed_path not in sys.path:
    sys.path.append(fixed_path)

modules_dir = f'{fixed_path}src/'
if modules_dir not in sys.path:
    sys.path.append(modules_dir)




experiment_path = f'{fixed_path}experiments/experiment_{whole_exp}/'
os.makedirs(experiment_path, exist_ok=True)

config_file_path = os.path.join(experiment_path, "train_clinical_config.yml")
with open(config_file_path, "r") as f:
    config = yaml.safe_load(f)

# --- Data ---
train_data_folder_path = config['train_data_folder_path']
extension = config['extension']
label_column = config['label_column']
remove_columns = config['remove_columns']
labels_level = config['labels_level']

# --- Seed parameters ---
starting_seed = config['starting_seed']

# --- Environment parameters ---
colab = config['colab']
save_memory = config['save_memory']

# --- Model Parameters ---
eval_metric = config['eval_metric']

n_patients = config['n_patients']


for key, value in config.items():
    print(f"{key}: {value}")


if save_memory:
    colab = False ##################################################à
    if colab:
        mem_path = '/content/temp_data'
    else:
        mem_path = f'{fixed_path}/temp_data'

    mem = MemorySave(base_dir = mem_path)
else:
    mem = None



In [ ]:

decache_files = ['src.shared.utils', 'src.shared.timepoints_elaboration',
                 'src.BinCellCNN.model', 'src.clinical']

# Remove from cache
from src.shared.utils import remove_from_cache
remove_from_cache(decache_files)

from src.shared.timepoints_elaboration import load_data, show_blast_distribution_perc
#from src.sample_level.show_results import show_blast_distribution_perc
from src.BinCellCNN.model import BinCellCNN
from src.clinical import generate_clinical_folds, tuning_and_train_model

# Single Cell Run

In [ ]:
%%time

all_datasets, all_datasets_labels, multiple_donations, samples_info_df = load_data(config = config,
                                                                data_folder_path = train_data_folder_path,
                                                                mem = mem)#,

if labels_level == 'cell':
    tot_perc_list = show_blast_distribution_perc(samples_info_df = samples_info_df,
                                                return_perc = True,
                                                decimals = 4,
                                                samples_ids = True)
    print(tot_perc_list)



In [ ]:
%%time

seed_manager = SeedManager(starting_seed=starting_seed)


train_kfolds = generate_clinical_folds(multiple_donations = multiple_donations,
                                  ALL_DATASETS= all_datasets,
                                  folds = 3,
                                  label_column = label_column,
                                  mem = mem,
                                  seed_manager = seed_manager)



In [ ]:
%%time

CV_models, best_thr = tuning_and_train_model(BinCellCnn = BinCellCNN,
                                                          config = config,
                                                          experiment_path = experiment_path,
                                                          train_kfolds = train_kfolds,
                                                          multiple_donations = multiple_donations,
                                                          ALL_DATASETS = all_datasets,
                                                          ALL_DATASETS_LABELS = all_datasets_labels,
                                                          mem = mem,
                                                          seed_manager = seed_manager)




In [ ]:
if mem:
    mem.close()
